# Category Model Extension (ModernBERT, Kaggle 2x T4)

Extends the **existing 10-class category model** (`model.safetensors` + `tokenizer.json` in the project root, gated `ferjaboss/ci-logs-classifier` repo) with **`train_real.csv`** (29k real log lines, 23 categories) **without losing the original 10 classes**.

### What it does
- Loads backbone + classifier weights from the local model files (head rows 0-9 preserved).
- Builds the **union label set**: 10 original + new from `train_real.csv` (33 total, minus rare-class filtering).
- Expands `classifier` Linear(768 -> N): old rows copied, new rows Xavier-initialised.
- **Phase 1 (head warmup)**: backbone frozen, train head only, higher LR.
- **Phase 2 (full fine-tune)**: unfreeze all, low LR, weighted CE + cosine schedule.
- **CV-style evaluation**: test report + confusion matrix + 5-fold inference aggregation (mean +/- std).

### Kaggle setup (2x T4)
- Upload **two** datasets via Add Input:
  1. `category-base-model`: `config.json`, `model.safetensors`, `tokenizer.json`, `tokenizer_config.json`, `special_tokens_map.json` (project root files).
  2. `cicd-selfhealing-dataset`: `pipeline-dataset/train_real.csv`.
- Or set `MODEL_DIR` / `DATA_CSV` explicitly in the config cell.
- fp16 (Turing T4; not bf16), gradient checkpointing, `accelerate.notebook_launcher` for 2-GPU data-parallel, effective batch kept constant across 1/2 GPUs.

In [ ]:
!pip install -q --force-reinstall --no-deps numpy==2.5.3 scipy==1.18.1
!pip install -q --upgrade transformers tokenizers accelerate datasets scikit-learn safetensors


In [ ]:
# CONFIG
import hashlib
import os
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

MAX_LEN = 256               # category model context (matches ml/category_model.py usage)
PER_DEVICE_BATCH = 16       # T4 16GB fits 16x256 easily; effective batch scaled below
GRAD_ACCUM = 2              # x2 GPUs -> effective batch 16*2*2 = 64
LR_HEAD = 5e-4              # head-only warmup LR (backbone frozen)
EPOCHS_HEAD = 1             # head warmup epochs
LR_FULL = 1e-5              # full fine-tune LR (low: preserve pretrained features)
EPOCHS_FULL = 3             # full fine-tune epochs
MIN_SAMPLES = 10            # drop train_real.csv classes with fewer rows (stratify needs >= ~10)
N_FOLDS = 5                 # CV-style inference folds (no retraining, eval only)
EVAL_BATCH = 64
MODEL_DIR = None            # explicit path to base-model dir (auto-detected if None)
DATA_CSV = None             # explicit path to train_real.csv (auto-detected if None)
OUTPUT_DIR = "/kaggle/working/category-extended"
HUB_REPO = ""                # set to push, e.g. "aziz123123v/cicd-log-category-extended"

print("category-extension config loaded")

# stale-kernel watchdog (same as rootcause_training.ipynb)
import importlib.metadata as _md
_installed = _md.version("numpy")
if np.__version__ != _installed:
        raise RuntimeError(
            f"KERNEL STALE: running numpy {np.__version__} != installed {_installed}. "
            "Restart the runtime, then run again."
        )

import scipy.sparse  # noqa: F401
from sklearn.model_selection import train_test_split  # noqa: F401
import torch  # noqa: F401
from transformers import AutoTokenizer, AutoConfig, AutoModelForSequenceClassification  # noqa: F401
import datasets  # noqa: F401
print("stack ok | numpy", np.__version__, "| scipy", scipy.__version__, "| torch", torch.__version__)

In [ ]:
# LOCATE base model files (gated repo -> local files uploaded as Kaggle dataset)
import glob as _glob

REQUIRED = ["config.json", "model.safetensors", "tokenizer.json"]
CANDIDATES = [
    MODEL_DIR,
    "/kaggle/input/category-base-model",
    "/kaggle/working/category-base-model",
]
# any dataset dir that contains model.safetensors
for _p in _glob.glob("/kaggle/input/**/model.safetensors", recursive=True):
    import os as _os
    CANDIDATES.append(_os.path.dirname(_p))

MODEL_DIR = next(
    (c for c in CANDIDATES if c and all(os.path.exists(os.path.join(c, f)) for f in REQUIRED)),
    None,
)
if MODEL_DIR is None:
    raise FileNotFoundError(
        "Base model files not found. Upload config.json + model.safetensors + tokenizer.json "
        "as a Kaggle dataset (Add Input), or set MODEL_DIR explicitly."
    )
print("MODEL_DIR:", MODEL_DIR)
print("files:", sorted(os.listdir(MODEL_DIR)))

cfg0 = AutoConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
_id2l = cfg0.id2label
EXISTING_LABELS = [_id2l[i] if i in _id2l else _id2l[str(i)] for i in range(cfg0.num_labels)]
print(f"existing labels ({len(EXISTING_LABELS)}):", EXISTING_LABELS)

In [ ]:
# DATASET: locate train_real.csv, filter rare classes, build UNION label set
import glob as _glob2

CANDS = [
    DATA_CSV,
    "/kaggle/input/cicd-selfhealing-dataset/train_real.csv",
    "/kaggle/working/train_real.csv",
]
CANDS += _glob2.glob("/kaggle/input/**/train_real.csv", recursive=True)
DATA_CSV = next((c for c in CANDS if c and os.path.exists(c)), None)
if DATA_CSV is None:
    raise FileNotFoundError("train_real.csv not found - attach the dataset in the Kaggle UI (Add Input).")
print("using", DATA_CSV)

df = pd.read_csv(DATA_CSV, encoding="utf-8")
df = df[["error_message", "category"]].dropna()
df = df[df["error_message"].str.strip() != ""].reset_index(drop=True)
print("base shape:", df.shape)
print("sources: n/a (train_real.csv) | categories:", df["category"].nunique())
print(df["category"].value_counts().to_string())

# drop ultra-rare classes that break stratified splits
counts = df["category"].value_counts()
rare = counts[counts < MIN_SAMPLES].index.tolist()
if rare:
    print(f"\ndropping rare classes (<{MIN_SAMPLES} rows):", rare)
    df = df[~df["category"].isin(rare)].reset_index(drop=True)
    print("kept shape:", df.shape)

NEW_LABELS = sorted(set(df["category"].unique()) - set(EXISTING_LABELS))
ALL_LABELS = EXISTING_LABELS + NEW_LABELS
label2id = {l: i for i, l in enumerate(ALL_LABELS)}
id2label = {i: l for i, l in enumerate(ALL_LABELS)}
print(f"\nunion labels: {len(EXISTING_LABELS)} original + {len(NEW_LABELS)} new = {len(ALL_LABELS)}")
print("new:", NEW_LABELS)

In [ ]:
# TOKENIZE + stratified train/val/test (80/10/10)
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)

idx = np.arange(len(df))
tr, tmp = train_test_split(idx, test_size=0.20, random_state=SEED, stratify=df["category"])
va, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=df.loc[tmp, "category"])
split_of = {}
split_of.update({i: "train" for i in tr})
split_of.update({i: "val" for i in va})
split_of.update({i: "test" for i in te})


def make_split(idxs):
    sub = df.loc[sorted(idxs), ["error_message", "category"]].reset_index(drop=True)
    ds = Dataset.from_pandas(sub)

    def tok(batch):
        enc = tokenizer(batch["error_message"], truncation=True, padding=False, max_length=MAX_LEN)
        return {
            "input_ids": enc["input_ids"],
            "attention_mask": enc["attention_mask"],
            "label": [label2id[c] for c in batch["category"]],
        }

    return ds.map(tok, batched=True, remove_columns=["error_message", "category"])


ds = {"train": make_split(tr), "val": make_split(va), "test": make_split(te)}
for s in ds:
    print(s, len(ds[s]))

# inverse-frequency class weights from TRAIN (tames Runtime/Test dominance)
_vc = df.loc[tr, "category"].value_counts()
_tot, _n = int(_vc.sum()), len(ALL_LABELS)
_w = []
for lab in ALL_LABELS:
    c = int(_vc.get(lab, 0))
    _w.append(_tot / (_n * c) if c else 0.0)
_w = torch.tensor(_w, dtype=torch.float32)
_w = _w / _w[_w > 0].mean()
CLASS_WEIGHTS = _w
print("class weights (min/max):", round(float(_w.min()), 3), round(float(_w.max()), 3))

In [ ]:
# MODEL: load base weights, expand classifier 10 -> N (old rows preserved)
import torch.nn as nn
from safetensors.torch import load_file


def build_extended_model():
    config = AutoConfig.from_pretrained(MODEL_DIR, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_config(config)
    model.load_state_dict(load_file(os.path.join(MODEL_DIR, "model.safetensors")))
    old = model.classifier  # Linear(768 -> 10)
    n_old, in_feat = old.weight.shape
    assert n_old == len(EXISTING_LABELS), (n_old, len(EXISTING_LABELS))
    new_head = nn.Linear(in_feat, len(ALL_LABELS))
    with torch.no_grad():
        new_head.weight[:n_old].copy_(old.weight)
        new_head.bias[:n_old].copy_(old.bias)
        nn.init.xavier_uniform_(new_head.weight[n_old:])
        new_head.bias[n_old:].zero_()
    model.classifier = new_head
    model.config.id2label = {str(i): l for i, l in enumerate(ALL_LABELS)}
    model.config.label2id = {l: i for i, l in enumerate(ALL_LABELS)}
    model.config.num_labels = len(ALL_LABELS)
    return model


# smoke test on CPU meta-build (weights load + shapes)
_m = build_extended_model()
print("classifier:", tuple(_m.classifier.weight.shape), "| num_labels:", _m.config.num_labels)
del _m

In [ ]:
# TRAINING (fp16 + grad checkpointing + 2-GPU DDP via accelerate notebook_launcher)
import torch
import torch.nn as nn

from sklearn.metrics import accuracy_score, f1_score
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments


class WeightedTrainer(Trainer):
    """CrossEntropy with inverse-frequency class weights (handles Runtime/Test dominance)."""

    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs["labels"]
        outputs = model(**{k: v for k, v in inputs.items() if k != "labels"})
        logits = outputs.logits
        w = None if self.class_weights is None else self.class_weights.to(logits.device)
        loss = nn.functional.cross_entropy(logits, labels.view(-1), weight=w)
        return (loss, outputs) if return_outputs else loss


def compute_metrics(ev):
    logits, y = ev
    p = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(y, p),
        "macro_f1": f1_score(y, p, average="macro", zero_division=0),
    }


def _mk_args(out, epochs, lr, grad_accum):
    n_gpu = max(1, torch.cuda.device_count())
    steps_per_epoch = max(1, len(ds["train"]) // (PER_DEVICE_BATCH * n_gpu * grad_accum))
    warmup = max(1, int(0.1 * epochs * steps_per_epoch))
    return TrainingArguments(
        output_dir=out,
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=EVAL_BATCH,
        gradient_accumulation_steps=grad_accum,
        learning_rate=lr,
        num_train_epochs=epochs,
        fp16=True,
        gradient_checkpointing=True,
        lr_scheduler_type="cosine",
        warmup_steps=warmup,
        logging_steps=25,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        seed=SEED,
        report_to=[],
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
    )


def train_fn(tr_ds, va_ds):
    model = build_extended_model()
    collator = DataCollatorWithPadding(tokenizer)
    # scale grad accumulation to ACTUAL device count so effective batch stays ~64
    grad_accum = (GRAD_ACCUM * 2) // max(1, torch.cuda.device_count())
    cw = CLASS_WEIGHTS

    # Phase 1: head warmup (backbone frozen) - learns new rows w/o destroying features
    for n, p in model.named_parameters():
        p.requires_grad = n.startswith("classifier") or n.startswith("head")
    t1 = WeightedTrainer(
        model=model, args=_mk_args("/kaggle/working/runs-phase1", EPOCHS_HEAD, LR_HEAD, grad_accum),
        train_dataset=tr_ds, eval_dataset=va_ds, data_collator=collator,
        compute_metrics=compute_metrics, class_weights=cw,
    )
    t1.train()

    # Phase 2: full fine-tune at low LR
    for p in model.parameters():
        p.requires_grad = True
    t2 = WeightedTrainer(
        model=model, args=_mk_args("/kaggle/working/runs", EPOCHS_FULL, LR_FULL, grad_accum),
        train_dataset=tr_ds, eval_dataset=va_ds, data_collator=collator,
        compute_metrics=compute_metrics, class_weights=cw,
    )
    t2.train()


# torch.cuda is never touched in the parent (forked children can't re-init CUDA); detect GPUs via nvidia-smi
import subprocess as _sp
_gpus = [l for l in _sp.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True, text=True).stdout.splitlines() if l.strip()]
print("GPUs detected:", _gpus or ["none"])

_is_kaggle = os.path.exists("/kaggle/working")
_2gpu_env = os.environ.get("CATEGORY_2GPU")
_allow_2gpu = (_2gpu_env == "1") if _2gpu_env is not None else (_is_kaggle and len(_gpus) >= 2)
print("platform:", "kaggle" if _is_kaggle else "other", "| 2-GPU:", _allow_2gpu)
# NCCL on Kaggle T4 x2: no NVLink/P2P -> default P2P init can hang; force PCIe/SHM
os.environ.setdefault("NCCL_P2P_DISABLE", "1")
os.environ.setdefault("NCCL_IB_DISABLE", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
if _allow_2gpu:
    from accelerate import notebook_launcher
    from torch.multiprocessing.spawn import ProcessRaisedException
    from torch.distributed.elastic.multiprocessing.errors import ChildFailedError
    try:
        notebook_launcher(train_fn, (ds["train"], ds["val"]), num_processes=len(_gpus))
    except (ChildFailedError, ProcessRaisedException):
        print("2-GPU launch failed on this host; retraining on ONE GPU, effective batch preserved")
        if not (os.environ.get("CUDA_VISIBLE_DEVICES") or "").strip():
            os.environ["CUDA_VISIBLE_DEVICES"] = "0"
        train_fn(ds["train"], ds["val"])
else:
    if not (os.environ.get("CUDA_VISIBLE_DEVICES") or "").strip():
        os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    print("training on ONE GPU (effective batch preserved via higher grad-accum)")
    train_fn(ds["train"], ds["val"])

In [ ]:
# TEST-SET EVALUATION (best checkpoint in /kaggle/working/runs) + CV-style 5-fold inference
import torch

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import pad_sequence

RUN_DIR = "/kaggle/working/runs"
import glob as _g, json as _j
_cands = sorted(_g.glob(os.path.join(RUN_DIR, "checkpoint-*")), key=lambda s: int(s.split("-")[-1]))
print("checkpoints:", _cands or ["none"])
_top_cfg = os.path.join(RUN_DIR, "config.json")
_src = RUN_DIR if os.path.exists(_top_cfg) else (_cands[-1] if _cands else RUN_DIR)
if not os.path.exists(os.path.join(_src, "config.json")):
    raise FileNotFoundError(f"No usable save in {RUN_DIR} - did Phase 2 finish? Check the training cell output.")
if _src != RUN_DIR:
    print("top-level save missing - using newest checkpoint:", _src)
_cfgp = os.path.join(_src, "config.json")
_cfg = _j.load(open(_cfgp, encoding="utf-8"))
if "model_type" not in _cfg:  # repair partial/interrupted save
    _cfg["model_type"] = "modernbert"
    _j.dump(_cfg, open(_cfgp, "w", encoding="utf-8"), indent=2)
    print("repaired config.json (added model_type=modernbert)")
model = AutoModelForSequenceClassification.from_pretrained(_src, trust_remote_code=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()
print("labels:", model.config.num_labels)


def _collate(batch):
    return {
        "input_ids": pad_sequence([torch.tensor(b["input_ids"]) for b in batch], batch_first=True),
        "attention_mask": pad_sequence([torch.tensor(b["attention_mask"]) for b in batch], batch_first=True),
        "labels": torch.tensor([b["label"] for b in batch]),
    }


def _predict(split):
    loader = DataLoader(split, batch_size=EVAL_BATCH, shuffle=False, collate_fn=_collate)
    preds, trues = [], []
    with torch.no_grad():
        for b in loader:
            logits = model(input_ids=b["input_ids"].to(device), attention_mask=b["attention_mask"].to(device)).logits
            preds.extend(logits.argmax(dim=-1).cpu().numpy())
            trues.extend(b["labels"].numpy())
    return np.array(preds), np.array(trues)


preds, trues = _predict(ds["test"])
print("test accuracy:", round(accuracy_score(trues, preds), 4),
      "macro_f1:", round(f1_score(trues, preds, average="macro", zero_division=0), 4))
print(classification_report(trues, preds, labels=list(range(len(ALL_LABELS))), target_names=ALL_LABELS, zero_division=0))
print("confusion matrix:\n", confusion_matrix(trues, preds, labels=list(range(len(ALL_LABELS)))))

# CV-style: same trained model, stratified 5-fold inference over val+test (no retraining)
import pandas as pd
pool_idx = np.array(list(range(len(ds["val"]))) )
# rebuild texts/labels for val+test from df splits is heavy; fold over the TEST split indices instead
y_all = np.array(ds["test"]["label"])
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_acc, fold_f1 = [], []
for f, (_, vidx) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
    sub = ds["test"].select(sorted(vidx.tolist()))
    p, t = _predict(sub)
    a = accuracy_score(t, p)
    m = f1_score(t, p, average="macro", zero_division=0)
    fold_acc.append(a)
    fold_f1.append(m)
    print(f"  fold {f}: acc={a:.4f} macro_f1={m:.4f}")
print(f"CV-inference mean acc: {np.mean(fold_acc):.4f} +- {np.std(fold_acc):.4f} | "
      f"macro_f1: {np.mean(fold_f1):.4f} +- {np.std(fold_f1):.4f}")

In [ ]:
# SAVE final model + tokenizer + label maps; optional PUSH TO HUB
import json as _json
import shutil as _sh

FINAL = "/kaggle/working/category-extended"
os.makedirs(FINAL, exist_ok=True)
RUN_DIR = "/kaggle/working/runs"
import glob as _g, json as _j
_cands = sorted(_g.glob(os.path.join(RUN_DIR, "checkpoint-*")), key=lambda s: int(s.split("-")[-1]))
print("checkpoints:", _cands or ["none"])
_top_cfg = os.path.join(RUN_DIR, "config.json")
_src = RUN_DIR if os.path.exists(_top_cfg) else (_cands[-1] if _cands else RUN_DIR)
if not os.path.exists(os.path.join(_src, "config.json")):
    raise FileNotFoundError(f"No usable save in {RUN_DIR} - did Phase 2 finish? Check the training cell output.")
if _src != RUN_DIR:
    print("top-level save missing - using newest checkpoint:", _src)
_cfgp = os.path.join(_src, "config.json")
_cfg = _j.load(open(_cfgp, encoding="utf-8"))
if "model_type" not in _cfg:  # repair partial/interrupted save
    _cfg["model_type"] = "modernbert"
    _j.dump(_cfg, open(_cfgp, "w", encoding="utf-8"), indent=2)
    print("repaired config.json (added model_type=modernbert)")
best = AutoModelForSequenceClassification.from_pretrained(_src, trust_remote_code=True)
best.save_pretrained(FINAL)
tokenizer.save_pretrained(FINAL)
with open(os.path.join(FINAL, "label_maps.json"), "w", encoding="utf-8") as f:
    _json.dump({"label2id": label2id, "id2label": {str(k): v for k, v in id2label.items()}}, f, indent=2)
# keep raw tokenizer.json alongside (category_model.py loads Tokenizer.from_file)
_raw = os.path.join(MODEL_DIR, "tokenizer.json")
if os.path.exists(_raw):
    _sh.copy(_raw, os.path.join(FINAL, "tokenizer.json"))
print("saved:", FINAL)

if HUB_REPO:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        try:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
            print("HF_TOKEN loaded from Kaggle secrets")
        except Exception as e:
            print("no Kaggle secret HF_TOKEN:", type(e).__name__)
    if not hf_token:
        print("HF_TOKEN not set - model stays local at", FINAL)
    else:
        best.push_to_hub(HUB_REPO, token=hf_token)
        tokenizer.push_to_hub(HUB_REPO, token=hf_token)
        print("pushed:", HUB_REPO)
else:
    print("HUB_REPO empty - model saved locally only (download category-extended NOW, /kaggle/working is wiped on session end)")

In [ ]:
# INFERENCE DEMO: original-class line + Jenkins tool lines (domain-shift check)
import torch

model = AutoModelForSequenceClassification.from_pretrained("/kaggle/working/category-extended", trust_remote_code=True)
tok = AutoTokenizer.from_pretrained("/kaggle/working/category-extended", trust_remote_code=True)
model.to(device).eval()
id2l = model.config.id2label
labels = [id2l[str(i)] if str(i) in id2l else id2l[i] for i in range(model.config.num_labels)]

demos = [
    "ERROR: Connection timed out after 30s while connecting to registry.example.com:443",
    "src/analyzer.py:1:1: F401 'os' imported but unused",
    ">> Issue: [B105:hardcoded_password_string] Possible hardcoded password: 'admin123'",
    "src/analyzer.py:27: error: Unsupported operand types for +  [operator]",
]
enc = tok(demos, truncation=True, padding=True, max_length=MAX_LEN, return_tensors="pt").to(device)
with torch.no_grad():
    probs = torch.softmax(model(**enc).logits, dim=-1)
for text, pr in zip(demos, probs):
    top = pr.topk(3)
    print("---", text[:90])
    for i, p in zip(top.indices.tolist(), top.values.tolist()):
        print(f"   {labels[i]:<24} {p:.3f}")

## Next steps
1. Download `category-extended` from the Kaggle output panel (right-click -> Download) - `/kaggle/working` is wiped on session end.
2. Point `ml/category_model.py` at the extended dir (or copy `model.safetensors` + `tokenizer.json` + `config.json` to project root).
3. Re-run `python -m ml.cross_val_eval --model category --category-model ./category_model_finetuned --folds 5` for the CV report on your data.
4. Rare classes (`Lint`, `Dependency`, `Performance`) will still be weak - collect more real examples or merge them into a `Lint`/`Other` bucket and re-run.